This notebook functions as the trial workspace for identifying skill gap between the job-seeker and real required skillset or job tasks.

In [59]:
!pip install -U -q \
  "google-genai" \
  langchain \
  langchain_community \
  langchain-core \
  langchain-google-genai \
  langchain_chroma \
  langchain-text-splitters \
  pypdf \
  chromadb \
  langchain-huggingface \
  sentence-transformers

In [60]:
# Import libraries
from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from google import genai
from langchain_chroma import Chroma
from dotenv import load_dotenv
import os

from pdfminer.high_level import extract_text
import re
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate, ChatPromptTemplate, PromptTemplate

In [61]:
# Load the `.env` file that contains secret variables like API Keys
load_dotenv()

# We set the Google API Key for authentication when using Google's AI models. This key is necessary to access the services and is kept secret to prevent unauthorized use.
# Ensure you keep the API Key secret in real projects to prevent others from stealing your quota.
os.environ['GEMINI_API_KEY'] = os.getenv('GEMINI_API_KEY')
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
client = genai.Client(api_key=GEMINI_API_KEY)

In [62]:
# General variable
EMBEDDING_MODEL = "models/gemini-embedding-2-preview"
CHAT_MODEL = "gemini-3.1-flash-lite-preview"
EMBEDDING_MODEL_HF = "sentence-transformers/all-MiniLM-L6-v2"
SPLITTER = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

In [63]:
# Embedding
## Using Gemini
embedding_model_gemini = GoogleGenerativeAIEmbeddings(
    google_api = GEMINI_API_KEY,
    model = EMBEDDING_MODEL
)

## Using Hugging Face (alternative)
embedding_model_hf = HuggingFaceEmbeddings(
  model_name=EMBEDDING_MODEL_HF,
  model_kwargs={'device': 'cpu'}, # Additional options to specify that the model should run on CPU instead of GPU.
  encode_kwargs={'normalize_embeddings': True} # Additional options to specify that the embeddings should be normalized (converted to unit vectors) after encoding.
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8263.29it/s]


In [64]:
embedding_model_gemini.embed_query("Halo dunia")

[-0.015651852,
 0.0225785,
 0.023220442,
 -0.0090757795,
 0.003642914,
 0.037259698,
 -0.004497421,
 -0.0041690064,
 -0.013463008,
 -0.049589597,
 -0.015484892,
 0.014773321,
 -0.020296104,
 0.011053799,
 0.008241014,
 -0.012337022,
 0.03022989,
 -0.008231969,
 -0.00782098,
 0.0034343058,
 0.007070554,
 0.012583276,
 -0.02876649,
 0.016649459,
 0.0027756314,
 0.015280596,
 0.014025338,
 -0.008428536,
 -0.011446846,
 0.114099205,
 -0.0027241276,
 -0.022779688,
 -0.013112678,
 -0.025243074,
 0.014475361,
 -0.00996546,
 -0.010431587,
 0.0057806713,
 0.0055379006,
 -0.029698525,
 0.022879709,
 0.014558149,
 0.016324246,
 -0.0011357066,
 0.013559143,
 -0.01769173,
 -0.02234921,
 -0.023883535,
 0.0044255364,
 0.0061957696,
 -0.010686217,
 -0.0016180571,
 0.0027808303,
 -0.0020056132,
 -0.023342848,
 -0.01495283,
 -0.0036039136,
 -0.010471661,
 -0.018822439,
 0.012384085,
 0.033220563,
 -0.0011620416,
 4.77314e-05,
 0.022708546,
 -0.013372056,
 0.0067976676,
 0.017165735,
 0.013233771,
 0.021

Processing CSV files

In [73]:
def csv_processing(file_path, meta_data_cols: list):
    """
    Load, clean, embed, and storing csv file to vector database.

    Arguments
        - file_path: CSV documents of job listings
        - meta_data_cols: Columns names that will be used for filtering 
    """
    loader_csv = CSVLoader(file_path, metadata_columns=meta_data_cols)
    docs = loader_csv.load()
    all_contents = [post for post in docs]
    chunks = SPLITTER.split_documents(all_contents)
    
    db_dynamic = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model_hf,
        persist_directory="./chroma_db/dynamic_csv"
    )
    print("Stored to vector db using Hugging Face.")
        
    print(f"Dynamic index updated: {len(chunks)} chunks")

CV Processing

In [74]:
chat_model = ChatGoogleGenerativeAI(
            google_api_key=GEMINI_API_KEY,
            model=CHAT_MODEL
        )

In [75]:
def extract_cv(cv_path: str) -> str:
    loader = extract_text(cv_path) # Load file

    # Clean file
    cv_text = loader.replace("\xa0", " ") # Remove non-breaking space
    cv_text = "\n".join(line.rstrip() for line in cv_text.splitlines()) # Remove trailing whitespace for each row
    cv_text = re.sub(r"\n{3,}", "\n\n", cv_text) # Two new lines maximum

    # Utilize LLM to structure results
    
    prompt = PromptTemplate.from_template("""
            Anda adalah AI yang berfungsi untuk mengekstrak informasi penting dari resume job-seeker yang ingin berkarir dibidang Data dan Artificial Intelligence. Berikan response dengan informasi yang ada tanpa penambahan atau pengurangan, dalam bentuk yang terstruktur dan rapi dengan format:
                            - Skill teknikal umum (contoh: programming, visualisasi data, dan lainnya) serta rating
                            - Tech stack yang dikuasai (serta rating)
                            - Soft skill (serta rating)
                            - Pengalaman kerja (role, durasi, industri)
                            - Pendidikan
                            - Sertifikasi
            
            CV:
            {cv_text}
        """)

    chain = prompt | chat_model

    response = chain.invoke({"cv_text": cv_text})

    return "\n".join(
            part["text"]
            for part in response.content
            if part["type"] == "text"
        )

In [76]:
cv_profile = extract_cv("dataset/CV_Prayoga Rasyid Sudrajat-1-2.pdf")

print(type(cv_profile))
print(repr(cv_profile[:200]))

<class 'str'>
'Berikut adalah ekstraksi informasi dari resume Prayoga Rasyid Sudrajat:\n\n**Skill Teknikal Umum**\n*   Data Analysis (4.5/5)\n*   Machine Learning (4/5)\n*   Data Communication (4/5)\n*   ETL Processes (4/'


Gap analysis between extracted CV and job openings

In [82]:
# Match pipeline
def match_cv_to_jobs(cv_path: str, columns_to_filter: list, value_to_extract: str):
    cv_profile = extract_cv(cv_path)
    csv_processing("dataset/AI Job Market Dataset.csv", meta_data_cols=columns_to_filter)
            
    # Load index  
    dynamic_store = Chroma(
        persist_directory="./chroma_db/dynamic_csv",
        embedding_function=embedding_model_hf
    )

    # Find matching job
    job_matches = dynamic_store.similarity_search(cv_profile, filter={"job_title": value_to_extract}, k=5)
    jobs_text = "\n\n---\n\n".join([d.page_content for d in job_matches]) # Formatting
 
    # LLM
    chat_model = ChatGoogleGenerativeAI(
            google_api_key=GEMINI_API_KEY,
            model=CHAT_MODEL
        )
    
    prompt = PromptTemplate.from_template("""
        Anda adalah AI Career Advisor.
        Jika informasi tidak tersedia pada dataset, katakan bahwa informasi tersebut tidak ditemukan dan jangan mengarang jawaban.
        
        PROFIL KANDIDAT:
        {cv_profile}

        LOWONGAN PEKERJAAN:
        {job_listings}
        
        Berikan:
        1. Title lowongan yang paling cocok dengan alasannya.
        2. Gap skill yang perlu dipenuhi oleh kandidat.
        3. Rekomendasi langkah karir selanjutnya.
    """)

    chain = prompt | chat_model
    response = chain.invoke({
        "cv_profile": cv_profile,
        "job_listings": jobs_text
    })

    return "\n".join(
                part["text"]
                for part in response.content
                if part["type"] == "text"
            )

In [83]:
# Check result
result = match_cv_to_jobs("dataset/CV_Prayoga Rasyid Sudrajat-1-2.pdf", columns_to_filter=["job_title"], value_to_extract="Data Scientist")

print(result)

Stored to vector db using Hugging Face.
Dynamic index updated: 10345 chunks
Sebagai AI Career Advisor, berikut adalah analisis kecocokan profil **Prayoga Rasyid Sudrajat** terhadap lowongan yang tersedia:

### 1. Title Lowongan yang Paling Cocok
Berdasarkan data yang diberikan, **tidak ada lowongan yang sepenuhnya cocok** dengan kualifikasi kandidat saat ini. 

*   **Analisis:** Ketiga lowongan yang tersedia (job_id: 2502, 1712, 7493) menetapkan syarat pendidikan **PhD** atau **Master**, sedangkan Prayoga adalah lulusan Sarjana (S1). Selain itu, persyaratan pengalaman kerja (3-11 tahun) jauh melampaui pengalaman yang dimiliki kandidat saat ini.
*   **Jika harus memilih yang paling mendekati secara skill:** **Job ID 1712** adalah yang paling teknis relevan karena mencakup *Python, SQL, ML, Deep Learning,* dan *Cloud*, di mana Prayoga memiliki kompetensi teknis yang kuat di bidang tersebut.

### 2. Gap Skill yang Perlu Dipenuhi
Jika Prayoga ingin menargetkan peran serupa di level *Entry/

# Coret-coret

In [ ]:
loader_csv = CSVLoader("dataset/AI Job Market Dataset.csv")
docs = loader_csv.load()
all_contents = [post for post in docs]

print(all_contents[0])
# for row in all_contents[:5]:
#     print(row.page_content)

page_content='job_id: 1
job_title: AI Engineer
company_size: Startup
company_industry: Retail
country: Canada
remote_type: Remote
experience_level: Senior
years_experience: 2
education_level: Master
skills_python: 0
skills_sql: 0
skills_ml: 0
skills_deep_learning: 1
skills_cloud: 0
salary: 158322
job_posting_month: 6
job_posting_year: 2024
hiring_urgency: Low
job_openings: 4' metadata={'source': 'dataset/AI Job Market Dataset.csv', 'row': 0}


In [ ]:
# Load index  
dynamic_store = Chroma(
    persist_directory="./chroma_db/dynamic_csv",
    embedding_function=embedding_model_hf
)

# Find matching job
job_matches = dynamic_store.similarity_search(query=cv_profile, k=5, filter={"job_title": "Data Scientist"})
jobs_text = "\n\n---\n\n".join([d.page_content for d in job_matches]) # Formatting

In [ ]:
print(jobs_text)

In [ ]:
# Ambil 5 dokumen tanpa filter untuk inspeksi
test_matches = dynamic_store.similarity_search(query=cv_profile, k=5)

for i, doc in enumerate(test_matches):
    print(f"Dokumen ke-{i+1}:")
    print(f"Metadata: {doc.metadata}")
    print(f"Isi: {doc.page_content[:100]}...\n")


Dokumen ke-1:
Metadata: {'source': 'dataset/AI Job Market Dataset.csv', 'row': 8803}
Isi: job_id: 8804
job_title: Machine Learning Engineer
company_size: Medium
company_industry: Technology
...

Dokumen ke-2:
Metadata: {'source': 'dataset/AI Job Market Dataset.csv', 'row': 8803}
Isi: job_id: 8804
job_title: Machine Learning Engineer
company_size: Medium
company_industry: Technology
...

Dokumen ke-3:
Metadata: {'row': 8803, 'source': 'dataset/AI Job Market Dataset.csv'}
Isi: job_id: 8804
job_title: Machine Learning Engineer
company_size: Medium
company_industry: Technology
...

Dokumen ke-4:
Metadata: {'row': 8803, 'source': 'dataset/AI Job Market Dataset.csv'}
Isi: job_id: 8804
job_title: Machine Learning Engineer
company_size: Medium
company_industry: Technology
...

Dokumen ke-5:
Metadata: {'source': 'dataset/AI Job Market Dataset.csv', 'row': 8304}
Isi: job_id: 8305
job_title: Machine Learning Engineer
company_size: Medium
company_industry: E-commerce
...

